# Project 3: Retail Demand Forecasting - V3 (God-Tier)

In V2, the Deep LSTM failed to predict volatile retail demand because Neural Networks struggle when they don't have explicit temporal context (like the day of the week, or what happened 7 days ago).

## V3 Upgrades: The $25,000 Kaggle Winner Strategy
We are abandoning the LSTM for volatile demand and implementing the actual architecture that won the Walmart M5 competition: **LightGBM with Feature Engineering**.

1. **Steady Demand**: SARIMAX
2. **Seasonal Demand**: Multiplicative Prophet
3. **Volatile Demand**: **LightGBM Regressor**. We engineer critical time-series features (`lag_1`, `lag_7`, `lag_28`, `rolling_mean_7`, and `day_of_week`). By giving the model explicit context, it will finally be able to predict intermittent demand spikes.

In [ ]:
!pip install prophet lightgbm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

### 1. Load the Data

In [ ]:
print("Loading M5 Dataset...")
df = pd.read_csv('/kaggle/input/m5-forecasting-accuracy/sales_train_validation.csv')

steady_item_id = 'FOODS_3_090_CA_1_validation'
seasonal_item_id = 'HOBBIES_1_234_CA_1_validation'
volatile_item_id = 'HOUSEHOLD_1_118_CA_1_validation'

steady_ts = df[df['id'] == steady_item_id].iloc[:, 6:].T
seasonal_ts = df[df['id'] == seasonal_item_id].iloc[:, 6:].T
volatile_ts = df[df['id'] == volatile_item_id].iloc[:, 6:].T

dates = pd.date_range(start='2011-01-29', periods=1913, freq='D')
steady_ts.index = dates
seasonal_ts.index = dates
volatile_ts.index = dates

steady_ts.columns = ['Sales']
seasonal_ts.columns = ['Sales']
volatile_ts.columns = ['Sales']

### 2. Advanced SARIMAX for Steady Demand

In [ ]:
train_steady = steady_ts[:-30]
test_steady = steady_ts[-30:]

print("Training Advanced SARIMAX...")
arima_model = ARIMA(train_steady, order=(5, 1, 2), seasonal_order=(1, 1, 1, 7))
arima_fit = arima_model.fit()

arima_preds = arima_fit.forecast(steps=30)

plt.figure(figsize=(10,4))
plt.plot(test_steady.index, test_steady['Sales'], label='Actual')
plt.plot(test_steady.index, arima_preds, color='red', label='SARIMAX Forecast')
plt.title('SARIMAX on Steady Demand (30-day forecast)')
plt.legend()
plt.show()

### 3. Multiplicative Prophet for Seasonal Demand

In [ ]:
prophet_df = seasonal_ts.reset_index()
prophet_df.columns = ['ds', 'y']
train_seasonal = prophet_df[:-30]
test_seasonal = prophet_df[-30:]

print("Training Advanced Multiplicative Prophet...")
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, seasonality_mode='multiplicative', changepoint_prior_scale=0.05)
m.fit(train_seasonal)

future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)

plt.figure(figsize=(10,4))
plt.plot(test_seasonal['ds'], test_seasonal['y'], label='Actual')
plt.plot(test_seasonal['ds'], forecast['yhat'][-30:], color='green', label='Multiplicative Prophet Forecast')
plt.title('Multiplicative Prophet on Seasonal Demand (30-day forecast)')
plt.legend()
plt.show()

### 4. LightGBM (The Kaggle Winner) for Volatile Demand
Instead of feeding raw sales to an LSTM, we engineer specific time-series features so the gradient boosting tree knows exactly what is happening historically.

In [ ]:
print("Engineering features for LightGBM...")
lgb_df = volatile_ts.copy()

# Date Features
lgb_df['day_of_week'] = lgb_df.index.dayofweek
lgb_df['is_weekend'] = lgb_df['day_of_week'].isin([5, 6]).astype(int)

# Lag Features (What happened yesterday? What happened exactly a week ago?)
lgb_df['lag_1'] = lgb_df['Sales'].shift(1)
lgb_df['lag_7'] = lgb_df['Sales'].shift(7)
lgb_df['lag_28'] = lgb_df['Sales'].shift(28)

# Rolling Features (What is the recent trend?)
lgb_df['rolling_mean_7'] = lgb_df['Sales'].shift(1).rolling(7).mean()

# Drop NaNs caused by shifting
lgb_df = lgb_df.dropna()

train_lgb = lgb_df[:-30]
test_lgb = lgb_df[-30:]

features = ['day_of_week', 'is_weekend', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7']
X_train = train_lgb[features]
y_train = train_lgb['Sales']
X_test = test_lgb[features]
y_test = test_lgb['Sales']

print("Training Kaggle-Winning LightGBM Model...")
model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

lgb_preds = model.predict(X_test)

plt.figure(figsize=(10,4))
plt.plot(test_lgb.index, y_test, label='Actual')
plt.plot(test_lgb.index, lgb_preds, color='purple', label='LightGBM Forecast')
plt.title('LightGBM (with Feature Engineering) on Volatile Demand (30-day forecast)')
plt.legend()
plt.show()